# Starter Notebook: Allen Institute Visual Cortex Calcium Imaging

**Dataset:** DANDI:000039 — *Contrast tuning in mouse visual cortex with calcium imaging*  
**Source:** https://dandiarchive.org/dandiset/000039  
**Modality:** Two-photon calcium imaging (GCaMP6s)  
**Species:** Mouse (*Mus musculus*)  
**Brain region:** Primary visual cortex (V1)  
**Data standard:** Neurodata Without Borders (NWB)

**Found with:** neural-search — a neuroscience dataset search engine  
**Verified:** `python scripts/eval/verify_demo_dataset.py dandi:000039`

---

## What this dataset contains

Allen Institute recordings of mouse V1 neurons using two-photon calcium imaging while presenting drifting gratings at multiple contrast levels. Suitable for:
- Replicating contrast response functions (CRF) in V1
- Studying population coding of stimulus contrast
- Benchmarking neural encoding models
- Testing spike deconvolution algorithms

## 1. Setup

Install dependencies (run once):

In [ ]:
# Uncomment to install
# !pip install dandi pynwb fsspec aiohttp remfile matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## 2. Browse the dataset

List available NWB files without downloading:

In [ ]:
from dandi.dandiapi import DandiAPIClient

dandiset_id = "000039"

with DandiAPIClient() as client:
    dandiset = client.get_dandiset(dandiset_id)
    print(f"Dandiset: {dandiset.get_metadata().name}")
    print(f"Version: {dandiset.version_id}\n")
    
    assets = list(dandiset.get_assets())
    print(f"Total NWB files: {len(assets)}")
    for asset in assets[:5]:
        size_mb = asset.size / 1e6
        print(f"  {asset.path}  ({size_mb:.1f} MB)")
    if len(assets) > 5:
        print(f"  ... and {len(assets) - 5} more")

## 3. Stream one session (no full download)

Use `remfile` + `h5py` to stream NWB data directly from DANDI:

In [ ]:
import remfile
import h5py
import pynwb

with DandiAPIClient() as client:
    dandiset = client.get_dandiset(dandiset_id)
    assets = list(dandiset.get_assets())
    # Pick the first NWB file
    asset = assets[0]
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)

# Stream without downloading
rfile = remfile.File(s3_url)
h5file = h5py.File(rfile, "r")
io = pynwb.NWBHDF5IO(file=h5file, load_namespaces=True)
nwb = io.read()

print("Session:", nwb.identifier)
print("Subject:", nwb.subject)
print("Session start:", nwb.session_start_time)

## 4. Inspect the NWB structure

In [ ]:
print("Acquisition keys:", list(nwb.acquisition.keys()))
print("Processing modules:", list(nwb.processing.keys()))
print("Intervals:", list(nwb.intervals.keys()) if nwb.intervals else "(none)")

## 5. Extract fluorescence traces (dF/F)

In [ ]:
# dF/F traces are in the ophys processing module
ophys = nwb.processing["ophys"]
dff_module = ophys.data_interfaces["DfOverF"]
response_series = list(dff_module.roi_response_series.values())[0]

# Shape: (n_timepoints, n_rois)
dff = response_series.data[:]   # loads into memory
timestamps = response_series.timestamps[:]

n_timepoints, n_rois = dff.shape
duration_min = (timestamps[-1] - timestamps[0]) / 60
print(f"dF/F matrix: {n_timepoints} timepoints × {n_rois} ROIs")
print(f"Duration: {duration_min:.1f} min")

## 6. Extract stimulus contrast levels

In [ ]:
# Stimulus table from intervals
stim_table = nwb.intervals["StimulusTable"]

contrasts = stim_table["contrast"].data[:]
start_times = stim_table["start_time"].data[:]
stop_times = stim_table["stop_time"].data[:]

unique_contrasts = np.unique(contrasts)
print(f"Contrast levels: {unique_contrasts}")
print(f"Number of trials: {len(contrasts)}")

## 7. Compute contrast response function (CRF)

In [ ]:
# For each trial, extract mean dF/F during stimulus window (0.5s post-onset)
RESPONSE_WINDOW = 0.5  # seconds
dt = np.median(np.diff(timestamps))
n_samples = int(RESPONSE_WINDOW / dt)

trial_responses = []  # (n_trials, n_rois)
for t_start in start_times:
    idx = np.searchsorted(timestamps, t_start)
    window = dff[idx : idx + n_samples, :]
    if len(window) == n_samples:
        trial_responses.append(window.mean(axis=0))

trial_responses = np.array(trial_responses)  # (n_trials, n_rois)
print(f"Trial response matrix: {trial_responses.shape}")

# Mean response per contrast for each ROI
mean_by_contrast = {c: [] for c in unique_contrasts}
for i, contrast in enumerate(contrasts[:len(trial_responses)]):
    mean_by_contrast[contrast].append(trial_responses[i].mean())  # avg across rois

crf_mean = [np.mean(mean_by_contrast[c]) for c in unique_contrasts]
crf_sem  = [np.std(mean_by_contrast[c]) / np.sqrt(len(mean_by_contrast[c])) for c in unique_contrasts]

## 8. Plot contrast response function

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.errorbar(
    unique_contrasts, crf_mean, yerr=crf_sem,
    marker='o', color='steelblue', capsize=4, linewidth=2,
)
ax.set_xscale('log')
ax.set_xlabel('Contrast', fontsize=12)
ax.set_ylabel('Mean dF/F', fontsize=12)
ax.set_title('Contrast Response Function — Mouse V1\n(DANDI:000039)', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Peak response at contrast {unique_contrasts[np.argmax(crf_mean)]:.2f}")

## 9. Next steps

This dataset is well-suited for:

| Goal | Approach |
|------|----------|
| Contrast sensitivity analysis | Fit Naka-Rushton model to CRF per neuron |
| Population coding | PCA / UMAP on trial-averaged responses |
| Spike deconvolution | Apply OASIS or CASCADE to dF/F traces |
| Cross-session comparison | Load multiple NWB files, match ROI types |
| Neural encoding model | Fit linear/nonlinear model to stimulus × response |

**Find related datasets with neural-search:**

```python
# From the neural-search repo:
# python -m neural_search.search query 'mouse calcium imaging visual cortex contrast tuning'
```

**Cite:** Allen Institute for Brain Science (2019). Visual Coding — Neuropixels and Calcium Imaging datasets. DANDI Archive. https://dandiarchive.org/dandiset/000039